Preprocessing section:

In [166]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [167]:
#reads in dataset, then chooses columns of interest
client_data = pd.read_csv("U_of_DE_Comp.csv")

In [186]:
columns = ["Acct1", "Acct3", "Acct5", "Col1", "Del4", "Del6", "bad_flg_90plus_24m"]
client_data = client_data[columns]
client_data.head()

,Acct1,Acct3,Acct5,Col1,Del4,Del6,bad_flg_90plus_24m
0,83,2,204,0,0,23,0
1,83,19,205,0,0,17,0
2,84,15,123,0,0,47,0
3,84,16,204,0,0,58,0
4,83,4,196,0,0,23,0


In [187]:
client_data.head()

,Acct1,Acct3,Acct5,Col1,Del4,Del6,bad_flg_90plus_24m
0,83,2,204,0,0,23,0
1,83,19,205,0,0,17,0
2,84,15,123,0,0,47,0
3,84,16,204,0,0,58,0
4,83,4,196,0,0,23,0


In [188]:
#describes dataset-- min, max, median, etc. 
client_data.describe()

,Acct1,Acct3,Acct5,Col1,Del4,Del6,bad_flg_90plus_24m
count,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000
mean,60.576490,7.713430,207.671390,0.491100,1.013220,59.307400,0.063073
std,26.513848,5.983374,131.823067,1.550031,7.612786,585.555557,0.243095
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,39.000000,3.000000,114.000000,0.000000,0.000000,23.000000,0.000000
50%,73.000000,6.000000,193.000000,0.000000,0.000000,23.000000,0.000000
75%,84.000000,11.000000,287.000000,0.000000,0.000000,23.000000,0.000000
max,84.000000,90.000000,9994.000000,77.000000,99.000000,9999.000000,1.000000


In [171]:
# the max value for acct1 is 99, which seems slightly inaccurate (see read me comments).
# replaces 99 values in acct1 with median value instead
acct1_clean_median = client_data.loc[client_data["Acct1"] != 99, "Acct1"].median()
print(acct1_clean_median)
client_data["Acct1"] = client_data["Acct1"].replace(99, acct1_clean_median)

73.0


In [189]:
# once again, max value for acct3 is 99-- seems off.
# replaces 99 values in acct3 with median value instead
acct3_clean_median = client_data.loc[client_data["Acct3"] != 99, "Acct3"].median()
print(acct3_clean_median)
client_data["Acct3"] = client_data["Acct3"].replace(99, acct3_clean_median)

6.0


In [190]:
# the max value for acct5 is 9999, which is not feasible.
# replaces 9999 values in acct5 with median value instead
acct5_clean_median = client_data.loc[client_data["Acct5"] != 9999, "Acct5"].median()
print(acct5_clean_median)
client_data["Acct5"] = client_data["Acct5"].replace(9999, acct5_clean_median)
# the max value for acc

193.0


In [191]:
# the max value for del6 is 9996, which is also not feasible.
# replaces 9996 values in del6 with median value instead
del6_clean_median = client_data.loc[client_data["Del6"] != 9996, "Del6"].median()
print(del6_clean_median)
client_data["Del6"] = client_data["Del6"].replace(9996, del6_clean_median)

23.0


In [192]:
client_data.head()

,Acct1,Acct3,Acct5,Col1,Del4,Del6,bad_flg_90plus_24m
0,83,2,204,0,0,23,0
1,83,19,205,0,0,17,0
2,84,15,123,0,0,47,0
3,84,16,204,0,0,58,0
4,83,4,196,0,0,23,0


Next steps: build decision tree. find and evaluate feature importances. 
Machine learning section:

In [193]:
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_curve

In [207]:
# defines independent and dependent variables
# splits the data in 30/70 split
# fits the decision tree using entropy as a criterion
# calculates predictions in classification report, and confusion matrix

X = client_data[["Acct1", "Acct3", "Acct5", "Col1", "Del4", "Del6"]]
y = client_data["bad_flg_90plus_24m"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size = 0.3,
    random_state = 42,
    stratify=y
)

tree = DecisionTreeClassifier(
    criterion="entropy",
    class_weight={0:1, 1:10},
    max_depth=3,
    min_samples_leaf=20,
    random_state=42
)

tree.fit(X_train, y_train)

y_pred = tree.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.96      0.86      0.91     84323
           1       0.20      0.53      0.30      5677

    accuracy                           0.84     90000
   macro avg       0.58      0.70      0.60     90000
weighted avg       0.92      0.84      0.87     90000

[[72562 11761]
 [ 2655  3022]]


In [206]:
# gives the relative importance for each predictor variable
# uses Gini impurity
importance = pd.DataFrame({
    "Variable": X.columns,
    "Importance": tree.feature_importances_
})

print(
    importance.sort_values(
        "Importance",
        ascending=False
    )
)

  Variable  Importance
3     Col1    0.585205
0    Acct1    0.289485
4     Del4    0.125311
1    Acct3    0.000000
2    Acct5    0.000000
5     Del6    0.000000


In [209]:
# proportion of default/not default observations
client_data["bad_flg_90plus_24m"].value_counts(normalize=True)

bad_flg_90plus_24m
0    0.936927
1    0.063073
Name: proportion, dtype: float64

In [ ]:
# random predictions, used as baseline
p = y_train.mean()  

np.random.seed(42)

y_random = np.random.choice(
    [0, 1],
    size=len(y_test),
    p=[1-p, p]
)

print(classification_report(y_test, y_random))

              precision    recall  f1-score   support

           0       0.94      0.94      0.94     84323
           1       0.07      0.06      0.07      5677

    accuracy                           0.88     90000
   macro avg       0.50      0.50      0.50     90000
weighted avg       0.88      0.88      0.88     90000

